In [ ]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsClassifier
from imblearn.under_sampling import RandomUnderSampler

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

# -------------------------------
# 0. 데이터 불러오기
X = pd.read_csv("X_std.csv")
y = pd.read_csv("y.csv").squeeze()

if set(y.unique()) == {1, 2}:
    y = y.map({1:0, 2:1}).astype(int)

print("데이터 로드 완료:", X.shape, y.shape, "Unique y:", y.unique())

# -------------------------------
# 1. 평가 함수
def evaluate_model(name, y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    f1 = f1_score(y_true, y_pred)
    acc = accuracy_score(y_true, y_pred)
    bal_acc = (recall + specificity) / 2
    auc = roc_auc_score(y_true, y_proba)
    return {
        "Algorithm": name,
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "Precision": precision,
        "Recall": recall,
        "Specificity": specificity,
        "F1-score": f1,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "AUC": auc
    }

# -------------------------------
# 2. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
X_test_scaled = scaler.transform(X_test).astype(np.float32)

# 언더샘플링
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42)
X_res, y_res = rus.fit_resample(X_train_scaled, y_train)

# -------------------------------
# 3. 파라미터 후보
param_grids = {
    "Logistic Regression": {'C': [0.1, 1, 10]},
    "Random Forest": {'n_estimators': [100, 200], 'max_depth': [5, 10]},
    "XGBoost": {'n_estimators': [100], 'max_depth': [3, 5]},
    "LightGBM": {'n_estimators': [100], 'num_leaves': [31, 50]},
    "CatBoost": {'depth': [4, 6], 'iterations': [100], 'learning_rate': [0.1]},
    "MLP": {'hidden_layer_sizes': [(64,), (64, 32)], 'alpha': [0.0001, 0.001]}
}

# -------------------------------
# 4. 기본 모델
base_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    "LightGBM": LGBMClassifier(random_state=42),
    "CatBoost": CatBoostClassifier(verbose=0, random_state=42),
    "MLP": MLPClassifier(max_iter=300, random_state=42)
}

# -------------------------------
# 5. 학습 + Test 평가
results = []

for name, model in base_models.items():
    clf = RandomizedSearchCV(
        model,
        param_distributions=param_grids[name],
        n_iter=2, cv=3, scoring='f1',
        n_jobs=2, random_state=42
    )
    clf.fit(X_res, y_res)
    best_model = clf.best_estimator_

    y_pred = best_model.predict(X_test_scaled)
    y_proba = best_model.predict_proba(X_test_scaled)[:, 1]

    results.append(evaluate_model(name, y_test, y_pred, y_proba))

# CNN (별도)
cnn = Sequential([
    Dense(64, activation='relu', input_shape=(X_res.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])
cnn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
cnn.fit(X_res, y_res, epochs=5, batch_size=32, verbose=0)

y_pred_cnn = cnn.predict(X_test_scaled).flatten()
y_pred_cnn_label = (y_pred_cnn >= 0.5).astype(int)
results.append(evaluate_model("CNN", y_test, y_pred_cnn_label, y_pred_cnn))

# KNN (별도)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_res, y_res)
y_pred_knn = knn.predict(X_test_scaled)
y_proba_knn = knn.predict_proba(X_test_scaled)[:, 1]
results.append(evaluate_model("KNN", y_test, y_pred_knn, y_proba_knn))

# -------------------------------
# 6. 결과 저장
result_df = pd.DataFrame(results)
result_df.to_csv("final_test_results.csv", index=False, encoding="utf-8-sig")

print("최종 Test 성능 저장 완료: final_test_results.csv")
print(result_df)
